<h1>Chapter 8 - Semantic Search and Retrieval-Augmented Generation</h1>
<i>Exploring a vital part of LLMs, search.</i>

# Dense Retrieval Example


## 1. Getting the text archive and chunking it


In [28]:
import os
import cohere

# Cohere API key
api_key = os.getenv("COHERE_API_KEY")

# Create and retrieve a Cohere API key from os.cohere.ai
co = cohere.Client(api_key)

In [29]:
text = """
The Witcher 3: Wild Hunt is a 2015 action role-playing game developed and published by CD Projekt. 
It is the sequel to the 2011 game The Witcher 2: Assassins of Kings and the third game in The Witcher video 
game series, played in an open world with a third-person perspective. The games follow the Witcher series of 
fantasy novels by Polish author Andrzej Sapkowski.

The game takes place in a fictional fantasy world based on Slavic folklore. Players control Geralt of Rivia, 
a monster slayer for hire known as a Witcher, and search for his adopted daughter who is on the run from the 
Wild Hunt. Players battle the game's many dangers with weapons and magic, interact with non-player characters, 
and complete quests to acquire experience points and gold, which are used to increase Geralt's abilities and 
purchase equipment. The game's story has three possible endings, determined by the player's choices at key 
points in the narrative. Development began in 2011 and lasted for three and a half years. Central and 
Northern European cultures formed the basis of the game's world. The game was developed using the 
REDengine 3, which enabled CD Projekt to create a complex story without compromising its open world. 
The music was primarily composed by Marcin Przybyłowicz and performed by the Brandenburg State Orchestra.

The Witcher 3: Wild Hunt was released for PlayStation 4, Windows, and Xbox One in May 2015, with a Nintendo 
Switch version released in October 2019, and PlayStation 5 and Xbox Series X/S versions (subtitled "Complete 
Edition") released in December 2022. The game received critical acclaim, with praise for its gameplay, 
narrative, world design, combat, and visuals, although it received minor criticism due to technical issues. 
It holds more than 200 game of the year awards and has been cited as one of the greatest video games ever 
made. Two expansions were also released to critical acclaim: Hearts of Stone and Blood and Wine. 
A "Game of the Year Edition" was released in August 2016, with the base game, expansions and all downloadable 
content included. The game has sold over 60 million units as of May 2025, making it one of the best-selling 
video games of all time. A sequel titled The Witcher IV is in development.
"""

# Split into a list of sentences
texts = text.split('.')

# Clean up to remove empty spaces and new lines
texts = [t.strip(' \n') for t in texts]

In [30]:
texts

['The Witcher 3: Wild Hunt is a 2015 action role-playing game developed and published by CD Projekt',
 'It is the sequel to the 2011 game The Witcher 2: Assassins of Kings and the third game in The Witcher video \ngame series, played in an open world with a third-person perspective',
 'The games follow the Witcher series of \nfantasy novels by Polish author Andrzej Sapkowski',
 'The game takes place in a fictional fantasy world based on Slavic folklore',
 'Players control Geralt of Rivia, \na monster slayer for hire known as a Witcher, and search for his adopted daughter who is on the run from the \nWild Hunt',
 "Players battle the game's many dangers with weapons and magic, interact with non-player characters, \nand complete quests to acquire experience points and gold, which are used to increase Geralt's abilities and \npurchase equipment",
 "The game's story has three possible endings, determined by the player's choices at key \npoints in the narrative",
 'Development began in 2011 

## 2. Embedding the Text Chunks


In [31]:
import numpy as np

# Get the embeddings
response = co.embed(
  texts=texts,
  input_type="search_document",
).embeddings

embeds = np.array(response)
print(embeds.shape)

(19, 4096)


## 3. Building The Search Index


In [32]:
import faiss

dim = embeds.shape[1]
index = faiss.IndexFlatL2(dim)
index.add(np.float32(embeds))

In [33]:
dim

4096

## 4. Search the index


In [34]:
import pandas as pd

def search(query, number_of_results=3):

  # 1. Get the query's embedding
  query_embed = co.embed(texts=[query],
                input_type="search_query",).embeddings[0]

  # 2. Retrieve the nearest neighbors
  distances , similar_item_ids = index.search(np.float32([query_embed]), number_of_results)

  # 3. Format the results
  texts_np = np.array(texts) # Convert texts list to numpy for easier indexing
  results = pd.DataFrame(data={'texts': texts_np[similar_item_ids[0]],
                              'distance': distances[0]})

  # 4. Print and return the results
  print(f"Query:'{query}'\nNearest neighbors:")
  return results

In [35]:
query = "whats the theme of the game?"
results = search(query)
print(results)

Query:'whats the theme of the game?'
Nearest neighbors:
                                               texts     distance
0  The game takes place in a fictional fantasy wo...  8342.869141
1  Players control Geralt of Rivia, \na monster s...  9007.980469
2  The game received critical acclaim, with prais...  9033.525391


In [36]:
import bm25s
import string
from sklearn.feature_extraction import _stop_words

def bm25s_tokenizer(text):
    tokenized_doc = []
    for token in text.lower().split():
        token = token.strip(string.punctuation)

        if len(token) > 0 and token not in _stop_words.ENGLISH_STOP_WORDS:
            tokenized_doc.append(token)
    return tokenized_doc

In [37]:
from tqdm import tqdm

tokenized_corpus = []
for passage in tqdm(texts):
    tokenized_corpus.append(bm25s_tokenizer(passage))

# Create BM25 index using bm25s and index it
bm25 = bm25s.BM25()
bm25.index(tokenized_corpus)

100%|██████████| 19/19 [00:00<00:00, 55924.05it/s]


BM25S Create Vocab:   0%|          | 0/19 [00:00<?, ?it/s]

BM25S Convert tokens to indices:   0%|          | 0/19 [00:00<?, ?it/s]

BM25S Count Tokens:   0%|          | 0/19 [00:00<?, ?it/s]

BM25S Compute Scores:   0%|          | 0/19 [00:00<?, ?it/s]

In [38]:
def keyword_search(query, top_k=3, num_candidates=15):
    print("Input question:", query)

    ##### BM25 search (lexical search) #####
    tokenized_query = bm25s_tokenizer(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

In [39]:
keyword_search(query = "whats the genre of the game")

Input question: whats the genre of the game
Top-3 lexical search (BM25) hits
	0.511	It is the sequel to the 2011 game The Witcher 2: Assassins of Kings and the third game in The Witcher video  game series, played in an open world with a third-person perspective
	0.483	A "Game of the Year Edition" was released in August 2016, with the base game, expansions and all downloadable  content included
	0.380	It holds more than 200 game of the year awards and has been cited as one of the greatest video games ever  made


## Caveats of Dense Retrieval


In [40]:
query = "how many awards did the game get?"
results = search(query)
results

Query:'how many awards did the game get?'
Nearest neighbors:


,texts,distance
0,It holds more than 200 game of the year awards...,6212.641113
1,"The game received critical acclaim, with prais...",7920.187500
2,The game has sold over 60 million units as of ...,8837.010742


# Reranking Example


In [41]:
query = "what is the genre of the game?"
results = co.rerank(query=query, documents=texts, top_n=3, return_documents=True)
results.results

[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The Witcher 3: Wild Hunt is a 2015 action role-playing game developed and published by CD Projekt'), index=0, relevance_score=0.22753839),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The game takes place in a fictional fantasy world based on Slavic folklore'), index=3, relevance_score=0.21457656),
 RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='The games follow the Witcher series of \nfantasy novels by Polish author Andrzej Sapkowski'), index=2, relevance_score=0.19337612)]

In [42]:
for idx, result in enumerate(results.results):
    print(idx, result.relevance_score , result.document.text)

0 0.22753839 The Witcher 3: Wild Hunt is a 2015 action role-playing game developed and published by CD Projekt
1 0.21457656 The game takes place in a fictional fantasy world based on Slavic folklore
2 0.19337612 The games follow the Witcher series of 
fantasy novels by Polish author Andrzej Sapkowski


In [43]:
def keyword_and_reranking_search(query, top_k=3, num_candidates=10):
    print("Input question:", query)

    ##### BM25s search (lexical search) #####
    tokenized_query = bm25s_tokenizer(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    top_n = np.argpartition(bm25_scores, -num_candidates)[-num_candidates:]
    bm25_hits = [{'corpus_id': idx, 'score': bm25_scores[idx]} for idx in top_n]
    bm25_hits = sorted(bm25_hits, key=lambda x: x['score'], reverse=True)

    print(f"Top-3 lexical search (BM25s) hits")
    for hit in bm25_hits[0:top_k]:
        print("\t{:.3f}\t{}".format(hit['score'], texts[hit['corpus_id']].replace("\n", " ")))

    #Add re-ranking
    docs = [texts[hit['corpus_id']] for hit in bm25_hits]

    print(f"\nTop-3 hits by rank-API ({len(bm25_hits)} BM25s hits re-ranked)")
    results = co.rerank(query=query, documents=docs, top_n=top_k, return_documents=True)
    for hit in results.results:
        print("\t{:.3f}\t{}".format(hit.relevance_score, hit.document.text.replace("\n", " ")))

In [44]:
keyword_and_reranking_search(query = "what is the genre of the game?")

Input question: what is the genre of the game?
Top-3 lexical search (BM25s) hits
	0.511	It is the sequel to the 2011 game The Witcher 2: Assassins of Kings and the third game in The Witcher video  game series, played in an open world with a third-person perspective
	0.483	A "Game of the Year Edition" was released in August 2016, with the base game, expansions and all downloadable  content included
	0.380	It holds more than 200 game of the year awards and has been cited as one of the greatest video games ever  made

Top-3 hits by rank-API (10 BM25s hits re-ranked)
	0.228	The Witcher 3: Wild Hunt is a 2015 action role-playing game developed and published by CD Projekt
	0.215	The game takes place in a fictional fantasy world based on Slavic folklore
	0.174	The game received critical acclaim, with praise for its gameplay,  narrative, world design, combat, and visuals, although it received minor criticism due to technical issues


# Retrieval-Augmented Generation

## Example: Grounded Generation with an LLM API


In [45]:
query = "awards obtained"

# 1- Retrieval
# We'll use embedding search. But ideally we'd do hybrid
results = search(query)

# 2- Grounded Generation
docs_dict = [{'text': text} for text in results['texts']]
response = co.chat(
    message = query,
    documents=docs_dict
)

print(response.text)

Query:'awards obtained'
Nearest neighbors:
The game has obtained more than 200 game of the year awards and has been cited as one of the greatest video games ever made.


In [46]:
print(response)

text='The game has obtained more than 200 game of the year awards and has been cited as one of the greatest video games ever made.' generation_id='ab59338f-facc-4c9a-90f7-6cdca75a48d4' citations=[ChatCitation(start=22, end=59, text='more than 200 game of the year awards', document_ids=['doc_0'], type='TEXT_CONTENT'), ChatCitation(start=73, end=124, text='cited as one of the greatest video games ever made.', document_ids=['doc_0'], type='TEXT_CONTENT')] documents=[{'id': 'doc_0', 'text': 'It holds more than 200 game of the year awards and has been cited as one of the greatest video games ever \nmade'}] is_search_required=None search_queries=None search_results=None finish_reason='COMPLETE' tool_calls=None chat_history=[Message_User(message='awards obtained', tool_calls=None, role='USER'), Message_Chatbot(message='The game has obtained more than 200 game of the year awards and has been cited as one of the greatest video games ever made.', tool_calls=None, role='CHATBOT')] prompt=None met

In [47]:
response.citations

[ChatCitation(start=22, end=59, text='more than 200 game of the year awards', document_ids=['doc_0'], type='TEXT_CONTENT'),
 ChatCitation(start=73, end=124, text='cited as one of the greatest video games ever made.', document_ids=['doc_0'], type='TEXT_CONTENT')]

## Example: RAG with Local Models


### Loading the Generation Model


In [48]:
#!wget https://huggingface.co/microsoft/Phi-3-mini-4k-instruct-gguf/resolve/main/Phi-3-mini-4k-instruct-q4.gguf

In [49]:
from langchain_community.chat_models import ChatLlamaCpp

model_path = os.getenv("MODEL_PHI_PATH")

llm = ChatLlamaCpp(
    model_path=model_path,
    temperature=0.7,
    max_tokens=500,
    n_ctx=2048, # Context window size
    seed=42,
    n_gpu_layers=-1, # Uncomment to use GPU acceleration
    verbose=False,
)

### Loading the Embedding Model

In [50]:
from langchain.embeddings.huggingface import HuggingFaceEmbeddings

# Embedding Model for converting text to numerical representations
embedding_model = HuggingFaceEmbeddings(
    model_name='BAAI/bge-small-en-v1.5'
)

/home/cairo/code/hands-on-llms/.venv/lib/python3.10/site-packages/huggingface_hub/file_download.py:942: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


### Preparing the Vector Database

In [51]:
from langchain.vectorstores import FAISS

# Create a local vector database
db = FAISS.from_texts(texts, embedding_model)

### The RAG Prompt


In [52]:
from langchain import PromptTemplate
from langchain.chains import RetrievalQA


# Create a prompt template
template = """<|user|>
Relevant information:
{context}

Provide a concise answer the following question using the relevant information provided above:
{question}<|end|>
<|assistant|>"""
prompt = PromptTemplate(
    template=template,
    input_variables=["context", "question"]
)

# RAG Pipeline
rag = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type='stuff',
    retriever=db.as_retriever(),
    chain_type_kwargs={
        "prompt": prompt
    },
    verbose=True
)

In [53]:
rag.invoke('awards obtained')



> Entering new RetrievalQA chain...

> Finished chain.


{'query': 'awards obtained',
 'result': ' The game has obtained more than 200 Game of the Year awards and has been cited as one of the greatest video games ever made.'}